Vamos a cargar los fact y los clientes

In [5]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
print(fact.count(), customers.count())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
1980 250


Ejercicio para comparar el Inner frente al left

In [6]:
inner = fact.join(customers, "customer_id", "inner")
left = fact.join(customers, "customer_id", "left")
print("inner", inner.count(), "left", left.count())

inner 1956 left 1980


Ahora vamos a sacar un MINUS

In [7]:
orphans = fact.join(customers, "customer_id", "left_anti")
orphans.select("order_id", "customer_id").distinct().orderBy("order_id").show()
print("líneas", orphans.count(), "pedidos", orphans.select("order_id").distinct().count())


+--------+-----------+
|order_id|customer_id|
+--------+-----------+
|  O00013|      CX013|
|  O00014|      CX014|
|  O00015|      CX015|
|  O00016|      CX016|
|  O00017|      CX017|
|  O00018|      CX018|
|  O00019|      CX019|
|  O00020|      CX020|
+--------+-----------+

líneas 24 pedidos 8


Mejora de codigo

In [8]:
orders = spark.read.parquet(str(STAGING / "orders_clean"))
print("orders", orders.count())
print("inner", orders.join(customers, "customer_id", "inner").count())  # 780
print("left ", orders.join(customers, "customer_id", "left").count())   # 788


orders 788
inner 780
left  788
